In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

c:\Users\geetikak\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# llm
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

Adding Checkpointer to remember the previous messages

In [3]:
@tool
def add(a: int, b: int):
    """Add two numbers."""
    return a + b

In [4]:
# list of exisitng tools
tools = [add]
llm_with_tools = llm.bind_tools(tools)

In [5]:
# Agent node
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(
        state["messages"]
    )
    return {
        "messages": [response]
    }

In [31]:
# Router
def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return END

In [32]:
# Build Graph-
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools)) # without creating def tools, this node was created via  langgraph.prebuilt's ToolNode.

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {
    "tools": "tools",
    END: END
})
builder.add_edge("tools", "agent")

In [33]:
# Add checkointer

checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer = checkpointer
)

In [34]:
# Conversation Identity

config1 = {
    "configurable":{
        "thread_id": "converstion-1"
    }
}
config2 = {
    "configurable": {
    "thread_id": "conversation-2"
    }
}

In [35]:
# FIRST CALL

res1 = graph.invoke({
    "messages": [{
        "role": "user",
        "content": "my name is Sam "
}]
},
config1)
print(res1['messages'][-1].content)

Nice to meet you, Sam! How can I assist you today?


In [36]:
# SECOND CALL
res2 = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is my name?"
            }
        ]
    }, config1
)

In [37]:
print("Thread1: ", res2['messages'][-1].content)

Thread1:  Your name is Sam.


In [38]:
res3 = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I have a Samsung phone."
            }
        ]
    }, config1
)

In [39]:
res4 = graph.invoke(
    {
        'messages': [{
            'role': 'user',
            'content': 'Which is the competitor brand of my phone?'
        }]
    }, config1
)

In [40]:
print(res4['messages'][-1])

content='The main competitor brands to Samsung’s smartphones are:\n\n| Competitor | Flagship Line(s) | Notable Strengths |\n|------------|------------------|-------------------|\n| **Apple** | iPhone (e.g., iPhone\u202f15, iPhone\u202f15\u202fPro) | Seamless hardware‑software integration, iOS ecosystem, strong privacy focus, premium design |\n| **Google** | Pixel (e.g., Pixel\u202f8, Pixel\u202f8\u202fPro) | Pure Android experience, fast updates, excellent computational photography, AI‑driven features |\n| **OnePlus** | OnePlus\u202f12, OnePlus\u202f12R | High‑performance specs at a slightly lower price, fast charging, OxygenOS (clean Android skin) |\n| **Xiaomi** | Mi, Redmi, POCO series | Aggressive pricing, large batteries, high‑refresh‑rate displays, strong value‑for‑money |\n| **Oppo / Realme** | Find X, Reno, Realme GT series | Innovative camera tech, fast charging, stylish designs, competitive pricing |\n| **Huawei** | P series, Mate series (though limited outside China) | Stron

Inspect the state

In [41]:
snapshot = graph.get_state(config1)
print(snapshot.values)

{'messages': [HumanMessage(content='my name is Sam ', additional_kwargs={}, response_metadata={}, id='866e7868-45b3-4c4a-a066-ec8937640293'), AIMessage(content='Nice to meet you, Sam! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "my name is Sam". Probably they want a response acknowledging. No request. Just respond.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 125, 'total_tokens': 171, 'completion_time': 0.096155792, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.0049628, 'prompt_tokens_details': None, 'queue_time': 0.282344266, 'total_time': 0.101118592}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f65c6-87bb-7e81-8841-6e1efa3f8709-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 46, 'total_tokens

In [42]:
print(snapshot.next)

()


In [43]:
for msg in snapshot.values["messages"]:
    print(
        type(msg).__name__,
        ":",
        msg.content
    )

HumanMessage : my name is Sam 
AIMessage : Nice to meet you, Sam! How can I assist you today?
HumanMessage : what is my name?
AIMessage : Your name is Sam.
HumanMessage : I have a Samsung phone.
AIMessage : Got it! Samsung phones have a lot of features and options. How can I help you with your device? Whether it’s troubleshooting, tips, app recommendations, or anything else, just let me know what you need.
HumanMessage : Which is the competitor brand of my phone?
AIMessage : The main competitor brands to Samsung’s smartphones are:

| Competitor | Flagship Line(s) | Notable Strengths |
|------------|------------------|-------------------|
| **Apple** | iPhone (e.g., iPhone 15, iPhone 15 Pro) | Seamless hardware‑software integration, iOS ecosystem, strong privacy focus, premium design |
| **Google** | Pixel (e.g., Pixel 8, Pixel 8 Pro) | Pure Android experience, fast updates, excellent computational photography, AI‑driven features |
| **OnePlus** | OnePlus 12, OnePlus 12R | High‑performa

Update state manually

invoke()
   = execute graph


get_state()
   = inspect graph state
   

update_state()
   = manually modify graph state

In [44]:
graph.update_state(
    config1,
    {
        "messages": [
            {
                "role": "user",
                "content": "Actually, call me Alex."
            }
        ]
    }
)

{'configurable': {'thread_id': 'converstion-1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18049b-5264-6902-800b-1decf7163a4a'}}

Now inspect again

In [45]:
snapshot = graph.get_state(config1)


for message in snapshot.values["messages"]:
    print(
        type(message).__name__,
        ":",
        message.content
    )

HumanMessage : my name is Sam 
AIMessage : Nice to meet you, Sam! How can I assist you today?
HumanMessage : what is my name?
AIMessage : Your name is Sam.
HumanMessage : I have a Samsung phone.
AIMessage : Got it! Samsung phones have a lot of features and options. How can I help you with your device? Whether it’s troubleshooting, tips, app recommendations, or anything else, just let me know what you need.
HumanMessage : Which is the competitor brand of my phone?
AIMessage : The main competitor brands to Samsung’s smartphones are:

| Competitor | Flagship Line(s) | Notable Strengths |
|------------|------------------|-------------------|
| **Apple** | iPhone (e.g., iPhone 15, iPhone 15 Pro) | Seamless hardware‑software integration, iOS ecosystem, strong privacy focus, premium design |
| **Google** | Pixel (e.g., Pixel 8, Pixel 8 Pro) | Pure Android experience, fast updates, excellent computational photography, AI‑driven features |
| **OnePlus** | OnePlus 12, OnePlus 12R | High‑performa

In [46]:
res5 = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What should you call me?"
            }
        ]
    },
    config1
)


print(
    res5["messages"][-1].content
)

I’ll call you Alex.


In [47]:
res6 = graph.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content":"what is my name?"
            }
        ]
    }, config2
)

print("Thread2: ", res3['messages'][-1].content)

Thread2:  Got it! Samsung phones have a lot of features and options. How can I help you with your device? Whether it’s troubleshooting, tips, app recommendations, or anything else, just let me know what you need.
